In [ ]:
# ── KAGGLE SETUP (only runs on Kaggle, no-op everywhere else) ───────────────
import os, sys
from pathlib import Path

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    import subprocess
    REPO_URL = "https://github.com/ltruonghai257/fake-new-detection.git"
    CLONE_DIR = Path("/kaggle/working/fake-news-detection")
    if not CLONE_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
        print(f"Cloned → {CLONE_DIR}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(CLONE_DIR / "requirements.txt")], check=True,
    )
    def _find_hdf5_root():
        for d in Path("/kaggle/input").rglob("processed_data/hdf5"):
            if d.is_dir(): return d.parent.parent
        return None
    hdf5_root = _find_hdf5_root()
    if hdf5_root is None:
        raise RuntimeError("processed_data/hdf5/ not found under /kaggle/input/")
    (CLONE_DIR / ".env.kaggle").write_text(f"DATA_ROOT={hdf5_root}\nPLATFORM=kaggle\n")
    print(f"DATA_ROOT={hdf5_root}")
else:
    print("Not Kaggle — setup skipped.")

In [ ]:
# ── Environment + path setup ────────────────────────────────────────────────
import os, sys
from pathlib import Path

def _detect_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
        return "kaggle"
    try:
        import google.colab; return "colab"
    except ImportError: pass
    if Path("/workspace").exists(): return "vastai"
    if sys.platform == "darwin": return "mac"
    return "windows"

PLATFORM = _detect_platform()

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
elif PLATFORM == "kaggle":
    PROJECT_ROOT = Path(os.environ.get("KAGGLE_PROJECT_ROOT", "/kaggle/working/fake-news-detection"))
else:
    PROJECT_ROOT = _nb_path.parents[1] if _nb_path.name.endswith(".ipynb") else Path.cwd().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
_env_file = PROJECT_ROOT / f".env.{PLATFORM}"
if not _env_file.exists(): _env_file = PROJECT_ROOT / ".env"
load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f"Platform : {PLATFORM}")
print(f"DATA_ROOT: {DATA_ROOT}  (exists={DATA_ROOT.exists()})")

# CAFE — Cross-modal Ambiguity Learning for Fake News Detection

> **This notebook trains the CAFE model only.** It is a fully independent architecture
> from COOLANT (`src/models/coolant_official.py`) — no COOLANT model class is imported
> or used anywhere below. The only thing shared with the COOLANT pipeline is the
> **raw pre-extracted feature files** (PhoBERT text embeddings + ResNet50 image
> embeddings, stored as `*.h5`) and the generic HDF5 loading utility that reads them —
> these are just feature caches, not part of any model.

Paper: *Cross-modal Ambiguity Learning for Multimodal Fake News Detection* (WWW 2022)  
Original repo: https://github.com/cyxanna/CAFE

**Training scheme (two tasks):**
```
Task 1 — Similarity:  L_SIM = CosineEmbeddingLoss(text_aligned, img_aligned, ±1)
                       Matched   = (text_i,   image_i)            → label +1
                       Unmatched = (text_i,   image_i+shift)      → label -1

Task 2 — Detection:   L_DET = L_CE(logits, det_label)
                       Real  = (text_i,   image_i)            → det_label 0
                       Fake  = (text_i,   image_i+shift)      → det_label 1
```

**Model classes used (all defined inline in this notebook, self-contained):**
`SimilarityModule_CAFE`, `DetectionModule_CAFE`, `EncodingPart_CAFE`, `FastCNN_CAFE`,
`AmbiguityLearning_CAFE`, `Encoder_CAFE`, `UnimodalDetection_CAFE`, `CrossModule4Batch_CAFE`.

**Architecture adaptations vs original paper:**
- `FastCNN` input dim: `200 → 768` (PhoBERT embed, no permute needed as data is `(B, 768, 128)`)
- Image encoder input: `512 → 2048` (ResNet50 instead of VGG16)
- Everything else (SimilarityModule, AmbiguityLearning, UnimodalDetection, CrossModule4Batch,
  DetectionModule) is identical to the original CAFE code.

**Label convention (this notebook):**
- Real (matched)   → detection label **0**
- Fake (unmatched) → detection label **1**

**Checkpoints/outputs** are written to `checkpoints_cafe/` (run name prefix `cafe_...`,
Kaggle dataset slug `cafe-best`) — fully separate from COOLANT's `checkpoints_coolant_v2/`.

In [ ]:
# ── CONFIG — edit this cell only ────────────────────────────────────────────
CONFIG = {
    "paths": {
        # Pre-extracted PhoBERT+ResNet50 feature caches (shared raw data, not a model).
        "train_hdf5": DATA_ROOT / "processed_data" / "hdf5" / "coolant_train.h5",
        "dev_hdf5":   DATA_ROOT / "processed_data" / "hdf5" / "coolant_dev.h5",
        "test_hdf5":  DATA_ROOT / "processed_data" / "hdf5" / "coolant_test.h5",
        # CAFE-specific checkpoint output — separate from COOLANT's checkpoints_coolant_v2/
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_cafe",
    },
    "model": {
        # Feature dims (must match HDF5 preprocessing output)
        "text_in_channels": 768,   # PhoBERT embed dim
        "image_input_dim":  2048,  # ResNet50 feature dim
        # CAFE architecture (original paper defaults)
        "cnn_channel":      32,
        "cnn_kernel_size": (1, 2, 4, 8),
        "shared_dim":      128,
        "sim_dim":          64,
        "feature_dim":      96,   # sim_dim + prime_dim*2 = 64 + 16 + 16
        "prime_dim":        16,
        "h_dim":            64,
    },
    "training": {
        "batch_size":             64,
        "max_epochs":            100,
        "negative_shift":          3,
        "min_batch_size":          4,
        "grad_clip":             1.0,
        "grad_accumulation_steps": 1,
        "warmup_epochs":           3,
        "lr":                   1e-3,
        "weight_decay":         0.0,   # original CAFE uses L2=0
        "seed":                   42,
        "sim_weight":            1.0,   # weight for L_SIM vs L_DET
        "early_stopping_patience": 15,   # stop if val_macro_f1 doesn't improve for N epochs
    },
    "checkpointing": {
        "selection_metric": "val_macro_f1",
        "checkpoint_every":  5,
    },
    "safety": {
        "smoke_test":            False,
        "smoke_batches":             2,
        "resume_from_checkpoint": None,
    },
}

# Kaggle: redirect writable outputs
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    CONFIG["paths"]["checkpoint_root"] = Path("/kaggle/working/checkpoints_cafe")

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)
print("CONFIG loaded — training CAFE model (independent from COOLANT).")
print(f"Checkpoints will be saved under: {CONFIG['paths']['checkpoint_root']}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import gc, json, random, math, shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, Independent
from torch.nn.functional import softplus
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# Generic raw-feature HDF5 loader (PhoBERT + ResNet50 caches, NOT the COOLANT model).
# This utility only reads pre-extracted (caption, image) tensors from disk; it has no
# dependency on the COOLANT architecture and is reused here purely for I/O convenience.
from src.preprocessing.coolant.pair_dataset import (
    create_coolant_dataloaders as create_feature_dataloaders,
)

print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── CAFE Model Definition (adapted for PhoBERT + ResNet50 features) ──────────
#
# All classes below are faithful to the original CAFE code (cyxanna/CAFE),
# with two targeted adaptations:
#   1. FastCNN_CAFE: in_channels=768 (PhoBERT), no permute (data already B,C,L)
#   2. EncodingPart_CAFE: image Linear 512→2048 (ResNet50 instead of VGG16)
# ─────────────────────────────────────────────────────────────────────────────

class FastCNN_CAFE(nn.Module):
    """CNN text encoder adapted for PhoBERT features (B, 768, seq_len).
    
    Original FastCNN expected (B, seq_len, 200) word embeddings and permuted
    to (B, 200, seq_len). Our data is already (B, 768, seq_len) — no permute needed.
    Output dim: channel * len(kernel_size) = 32*4 = 128 (same as original).
    """
    def __init__(self, in_channels=768, channel=32, kernel_size=(1, 2, 4, 8)):
        super().__init__()
        self.fast_cnn = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, channel, kernel_size=k),
                nn.BatchNorm1d(channel),
                nn.ReLU(),
                nn.AdaptiveMaxPool1d(1)
            )
            for k in kernel_size
        ])

    def forward(self, x):
        # x: (B, 768, seq_len) — already (B, C, L), no permute needed
        return torch.cat([m(x).squeeze(2) for m in self.fast_cnn], dim=1)  # (B, 128)


class EncodingPart_CAFE(nn.Module):
    """Shared encoder for text (PhoBERT) and image (ResNet50) features.
    
    Adaptation: image Linear changed from 512→256 to 2048→256.
    Text CNN channel changed from 200 to 768.
    """
    def __init__(
        self,
        text_in_channels=768,
        image_input_dim=2048,
        cnn_channel=32,
        cnn_kernel_size=(1, 2, 4, 8),
        shared_image_dim=128,
        shared_text_dim=128,
    ):
        super().__init__()
        self.shared_text_encoding = FastCNN_CAFE(
            in_channels=text_in_channels,
            channel=cnn_channel,
            kernel_size=cnn_kernel_size,
        )  # output: cnn_channel * len(kernel_size) = 128
        self.shared_text_linear = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(64, shared_text_dim),
            nn.BatchNorm1d(shared_text_dim),
            nn.ReLU()
        )
        self.shared_image = nn.Sequential(
            nn.Linear(image_input_dim, 256),   # 2048 instead of 512
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(256, shared_image_dim),
            nn.BatchNorm1d(shared_image_dim),
            nn.ReLU()
        )

    def forward(self, text, image):
        text_encoding = self.shared_text_encoding(text)   # (B, 128)
        text_shared   = self.shared_text_linear(text_encoding)  # (B, 128)
        image_shared  = self.shared_image(image)                # (B, 128)
        return text_shared, image_shared


# ── The remaining modules are verbatim from cyxanna/CAFE (model.py) ──────────

class SimilarityModule_CAFE(nn.Module):
    def __init__(self, text_in_channels=768, image_input_dim=2048,
                 cnn_channel=32, cnn_kernel_size=(1,2,4,8),
                 shared_dim=128, sim_dim=64):
        super().__init__()
        self.encoding = EncodingPart_CAFE(
            text_in_channels=text_in_channels,
            image_input_dim=image_input_dim,
            cnn_channel=cnn_channel,
            cnn_kernel_size=cnn_kernel_size,
            shared_image_dim=shared_dim,
            shared_text_dim=shared_dim,
        )
        self.text_aligner = nn.Sequential(
            nn.Linear(shared_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, sim_dim),
            nn.BatchNorm1d(sim_dim),
            nn.ReLU()
        )
        self.image_aligner = nn.Sequential(
            nn.Linear(shared_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, sim_dim),
            nn.BatchNorm1d(sim_dim),
            nn.ReLU()
        )
        self.sim_classifier = nn.Sequential(
            nn.BatchNorm1d(sim_dim * 2),
            nn.Linear(sim_dim * 2, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, text, image):
        text_enc, image_enc = self.encoding(text, image)
        text_aligned  = self.text_aligner(text_enc)    # (B, sim_dim)
        image_aligned = self.image_aligner(image_enc)  # (B, sim_dim)
        sim_feature   = torch.cat([text_aligned, image_aligned], dim=1)
        pred_sim      = self.sim_classifier(sim_feature)
        return text_aligned, image_aligned, pred_sim


class Encoder_CAFE(nn.Module):
    """Variational encoder producing a Normal distribution (z_dim=2)."""
    def __init__(self, z_dim=2):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(True),
            nn.Linear(64, z_dim * 2),
        )

    def forward(self, x):
        params = self.net(x)
        mu, sigma = params[:, :self.z_dim], params[:, self.z_dim:]
        sigma = softplus(sigma) + 1e-7
        return Independent(Normal(loc=mu, scale=sigma), 1)


class AmbiguityLearning_CAFE(nn.Module):
    """Symmetric KL divergence between text and image latent distributions."""
    def __init__(self):
        super().__init__()
        self.encoder_text  = Encoder_CAFE()
        self.encoder_image = Encoder_CAFE()

    def forward(self, text_encoding, image_encoding):
        p_z1 = self.encoder_text(text_encoding)
        p_z2 = self.encoder_image(image_encoding)
        z1 = p_z1.rsample()
        z2 = p_z2.rsample()
        kl_1_2 = p_z1.log_prob(z1) - p_z2.log_prob(z1)
        kl_2_1 = p_z2.log_prob(z2) - p_z1.log_prob(z2)
        skl = (kl_1_2 + kl_2_1) / 2.0
        skl = torch.sigmoid(skl)  # (B,)
        return skl


class UnimodalDetection_CAFE(nn.Module):
    """Per-modality unimodal representation (shared_dim → prime_dim)."""
    def __init__(self, shared_dim=128, prime_dim=16):
        super().__init__()
        self.text_uni = nn.Sequential(
            nn.Linear(shared_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, prime_dim),
            nn.BatchNorm1d(prime_dim),
            nn.ReLU()
        )
        self.image_uni = nn.Sequential(
            nn.Linear(shared_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.ReLU(),
            nn.Linear(shared_dim, prime_dim),
            nn.BatchNorm1d(prime_dim),
            nn.ReLU()
        )

    def forward(self, text_enc, image_enc):
        return self.text_uni(text_enc), self.image_uni(image_enc)  # (B, 16) each


class CrossModule4Batch_CAFE(nn.Module):
    """Cross-modal correlation via outer product + adaptive pooling."""
    def __init__(self, corre_out_dim=64):
        super().__init__()
        self.softmax = nn.Softmax(-1)
        self.pooling = nn.AdaptiveMaxPool1d(1)
        self.c_specific = nn.Sequential(
            nn.Linear(64, corre_out_dim),
            nn.BatchNorm1d(corre_out_dim),
            nn.ReLU()
        )

    def forward(self, text, image):
        # text: (B, 64), image: (B, 64)
        corre_dim = text.shape[1]
        sim = torch.matmul(text.unsqueeze(2), image.unsqueeze(1)) / math.sqrt(corre_dim)
        corr = self.softmax(sim)
        corr_p = self.pooling(corr).squeeze(2)  # (B, 64)
        return self.c_specific(corr_p)          # (B, corre_out_dim)


class DetectionModule_CAFE(nn.Module):
    """Detection module: combines unimodal + ambiguity-weighted cross-modal features."""
    def __init__(self, text_in_channels=768, image_input_dim=2048,
                 cnn_channel=32, cnn_kernel_size=(1,2,4,8),
                 shared_dim=128, prime_dim=16, sim_dim=64,
                 feature_dim=96, h_dim=64):
        super().__init__()
        # Separate EncodingPart instance (same as original CAFE's DetectionModule)
        self.encoding = EncodingPart_CAFE(
            text_in_channels=text_in_channels,
            image_input_dim=image_input_dim,
            cnn_channel=cnn_channel,
            cnn_kernel_size=cnn_kernel_size,
            shared_image_dim=shared_dim,
            shared_text_dim=shared_dim,
        )
        self.ambiguity_module = AmbiguityLearning_CAFE()
        self.uni_repre        = UnimodalDetection_CAFE(shared_dim=shared_dim, prime_dim=prime_dim)
        self.cross_module     = CrossModule4Batch_CAFE(corre_out_dim=sim_dim)
        self.classifier_corre = nn.Sequential(
            nn.Linear(feature_dim, h_dim),
            nn.BatchNorm1d(h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),
            nn.BatchNorm1d(h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, 2)
        )

    def forward(self, text_raw, image_raw, text_aligned, image_aligned):
        """Args:
            text_raw, image_raw:       original features (for unimodal encoding)
            text_aligned, image_aligned: 64-dim from SimilarityModule (for cross + ambiguity)
        """
        skl = self.ambiguity_module(text_aligned, image_aligned)      # (B,)
        text_prime, image_prime = self.encoding(text_raw, image_raw)  # (B, 128) each
        text_prime, image_prime = self.uni_repre(text_prime, image_prime)  # (B, 16) each
        correlation = self.cross_module(text_aligned, image_aligned)  # (B, 64)

        weight_uni   = (1 - skl).unsqueeze(1)  # low ambiguity → rely on unimodal
        weight_corre = skl.unsqueeze(1)         # high ambiguity → rely on cross-modal

        text_final  = weight_uni   * text_prime    # (B, 16)
        img_final   = weight_uni   * image_prime   # (B, 16)
        corre_final = weight_corre * correlation   # (B, 64)

        final = torch.cat([text_final, img_final, corre_final], dim=1)  # (B, 96)
        return self.classifier_corre(final)  # (B, 2)


print("CAFE model classes defined.")

In [ ]:
# ── Device, seed, dataloaders (raw features only — CAFE model built in next cell) ─
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu")
seed_everything(CONFIG["training"]["seed"])
print(f"Device: {DEVICE}")

loaders, datasets = create_feature_dataloaders(
    str(CONFIG["paths"]["train_hdf5"]),
    str(CONFIG["paths"]["dev_hdf5"]),
    str(CONFIG["paths"]["test_hdf5"]),
    batch_size=CONFIG["training"]["batch_size"],
)

if CONFIG["safety"]["smoke_test"]:
    from itertools import islice
    class _SmokeLoader:
        def __init__(self, l, n): self._l, self._n = l, n
        def __iter__(self): return islice(self._l, self._n)
        def __len__(self): return min(self._n, len(self._l))
    loaders = {k: _SmokeLoader(v, CONFIG["safety"]["smoke_batches"]) for k, v in loaders.items()}
    print(f"SMOKE TEST: {CONFIG['safety']['smoke_batches']} batches/split")

# Verify shapes
_cap, _img, _ = next(iter(loaders["train"]))
print(f"caption: {tuple(_cap.shape)}  (B, embed=768, seq=128)  [PhoBERT features]")
print(f"image:   {tuple(_img.shape)}  (B, 2048)  [ResNet50 features]")
print("These are raw feature tensors — the CAFE model is built and trained below.")

In [ ]:
# ── Build CAFE modules ───────────────────────────────────────────────────────
m_cfg = CONFIG["model"]

similarity_module = SimilarityModule_CAFE(
    text_in_channels=m_cfg["text_in_channels"],
    image_input_dim=m_cfg["image_input_dim"],
    cnn_channel=m_cfg["cnn_channel"],
    cnn_kernel_size=m_cfg["cnn_kernel_size"],
    shared_dim=m_cfg["shared_dim"],
    sim_dim=m_cfg["sim_dim"],
).to(DEVICE)

detection_module = DetectionModule_CAFE(
    text_in_channels=m_cfg["text_in_channels"],
    image_input_dim=m_cfg["image_input_dim"],
    cnn_channel=m_cfg["cnn_channel"],
    cnn_kernel_size=m_cfg["cnn_kernel_size"],
    shared_dim=m_cfg["shared_dim"],
    prime_dim=m_cfg["prime_dim"],
    sim_dim=m_cfg["sim_dim"],
    feature_dim=m_cfg["feature_dim"],
    h_dim=m_cfg["h_dim"],
).to(DEVICE)

sim_params = sum(p.numel() for p in similarity_module.parameters())
det_params = sum(p.numel() for p in detection_module.parameters())
print(f"SimilarityModule parameters: {sim_params:,}")
print(f"DetectionModule  parameters: {det_params:,}")
print(f"Total parameters:            {sim_params + det_params:,}")

In [ ]:
# ── Losses, optimizers, schedulers ──────────────────────────────────────────
loss_func_sim = nn.CosineEmbeddingLoss()  # Task 1: similarity
loss_func_det = nn.CrossEntropyLoss()     # Task 2: detection

lr = CONFIG["training"]["lr"]
wd = CONFIG["training"]["weight_decay"]
optim_sim = torch.optim.Adam(similarity_module.parameters(), lr=lr, weight_decay=wd)
optim_det = torch.optim.Adam(detection_module.parameters(), lr=lr, weight_decay=wd)

def _make_scheduler(opt):
    wu = CONFIG["training"]["warmup_epochs"]
    T  = CONFIG["training"]["max_epochs"]
    def _lr_lambda(ep):
        if ep < wu: return ep / max(1, wu)
        p = (ep - wu) / max(1, T - wu)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda)

sched_sim = _make_scheduler(optim_sim)
sched_det = _make_scheduler(optim_det)

print(f"Optimizers: Adam (lr={lr}, wd={wd})")
print(f"Scheduler: warmup={CONFIG['training']['warmup_epochs']} epochs, then cosine")

In [ ]:
# ── Pair construction helpers ─────────────────────────────────────────────────

def prepare_sim_data(text, image, shift):
    """Create matched and unmatched pairs for the similarity task (Task 1).
    
    All samples are treated as genuine matched pairs (real news).
    Unmatched pairs are created by rolling image along batch dim.
    
    Returns:
        fixed_text:      (B, 768, 128) — same text for both branches
        matched_image:   (B, 2048)     — original image
        unmatched_image: (B, 2048)     — rolled image
    """
    return text.clone(), image.clone(), image.roll(shifts=shift, dims=0)


def make_detection_batch(text, image, shift):
    """Create detection batch: real (matched) + fake (unmatched).
    
    Real  = original (text_i, image_i)          → label 0
    Fake  = mismatched (text_i, image_{i+shift}) → label 1
    
    Returns:
        det_text:   (2B, 768, 128)
        det_image:  (2B, 2048)
        det_labels: (2B,) long
    """
    B = text.size(0)
    det_text  = torch.cat([text,  text], dim=0)
    det_image = torch.cat([image, image.roll(shift, dims=0)], dim=0)
    det_labels = torch.cat([
        torch.zeros(B, dtype=torch.long, device=text.device),  # 0 = real
        torch.ones( B, dtype=torch.long, device=text.device),  # 1 = fake
    ], dim=0)
    return det_text, det_image, det_labels


# ── Sanity check (one batch) ─────────────────────────────────────────────────
similarity_module.train()
detection_module.train()

_cap, _img, _ = next(iter(loaders["train"]))
_cap, _img = _cap.to(DEVICE), _img.to(DEVICE)

# Task 1: Similarity
_ft, _img_m, _img_u = prepare_sim_data(_cap, _img, CONFIG["training"]["negative_shift"])
_ta_m, _ia_m, _ = similarity_module(_ft, _img_m)
_ta_u, _ia_u, _ = similarity_module(_ft, _img_u)
_ta_all = torch.cat([_ta_m, _ta_u], dim=0)
_ia_all = torch.cat([_ia_m, _ia_u], dim=0)
_lbl_cos = torch.cat([
    torch.ones(_ta_m.size(0), device=DEVICE),
    -torch.ones(_ta_u.size(0), device=DEVICE)
], dim=0)
_l_sim = loss_func_sim(_ta_all, _ia_all, _lbl_cos)

# Task 2: Detection
_dc, _di, _dl = make_detection_batch(_cap, _img, CONFIG["training"]["negative_shift"])
with torch.no_grad():
    _ta_d, _ia_d, _ = similarity_module(_dc, _di)
_det_logits = detection_module(_dc, _di, _ta_d, _ia_d)
_l_det = loss_func_det(_det_logits, _dl)

print(f"L_SIM  = {_l_sim.item():.4f}  {'OK' if torch.isfinite(_l_sim) else 'NaN!'}")
print(f"L_DET  = {_l_det.item():.4f}  {'OK' if torch.isfinite(_l_det) else 'NaN!'}")
print(f"det_logits shape: {_det_logits.shape}  (2B=2×{_cap.size(0)}, 2 classes)")
print("Sanity check passed.")

In [ ]:
# ── Run directory + checkpoint helpers ──────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name  = f"cafe_{timestamp}"
run_dir   = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")


def save_checkpoint(path, epoch, history, metrics):
    torch.save({
        "similarity_module_state": similarity_module.state_dict(),
        "detection_module_state":  detection_module.state_dict(),
        "optim_sim_state":         optim_sim.state_dict(),
        "optim_det_state":         optim_det.state_dict(),
        "epoch":   epoch,
        "metrics": metrics,
        "training_history": history,
        "config": CONFIG,
    }, path)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    similarity_module.load_state_dict(ckpt["similarity_module_state"])
    detection_module.load_state_dict(ckpt["detection_module_state"])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}: {path}")
    return ckpt


start_epoch = 0
if CONFIG["safety"]["resume_from_checkpoint"]:
    _ckpt = load_checkpoint(CONFIG["safety"]["resume_from_checkpoint"])
    start_epoch = _ckpt["epoch"] + 1

# ── Kaggle auto-upload best model ────────────────────────────────────────────
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists())
_KAGGLE_API = None
_KAGGLE_UPLOAD_OK = False

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _sec = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"]      = _sec.get_secret("KAGGLE_KEY")
        from kaggle.api.kaggle_api_extended import KaggleApi
        _KAGGLE_API = KaggleApi(); _KAGGLE_API.authenticate()
        _KAGGLE_UPLOAD_OK = True
        print(f"Kaggle API authenticated as {os.environ['KAGGLE_USERNAME']}")
    except Exception as e:
        print(f"Kaggle secrets/API not set up ({e}) — best model saved to Output tab only.")

_BEST_DS_SLUG = "cafe-best"


def kaggle_upload_best(ckpt_path, epoch, f1):
    if not _KAGGLE_UPLOAD_OK:
        return
    try:
        staging = Path("/kaggle/working/_ds_best_cafe")
        staging.mkdir(exist_ok=True)
        for old in staging.glob("*.pth"): old.unlink()
        shutil.copy2(ckpt_path, staging / "best_macro_f1.pth")
        (staging / "best_info.json").write_text(json.dumps({
            "run": run_name, "epoch": int(epoch), "val_macro_f1": float(f1),
            "saved_at": datetime.now().isoformat(),
        }, indent=2))
        (staging / "dataset-metadata.json").write_text(json.dumps({
            "title": "CAFE Best Model",
            "id": f"{os.environ['KAGGLE_USERNAME']}/{_BEST_DS_SLUG}",
            "licenses": [{"name": "other"}],
        }))
        try:
            _KAGGLE_API.dataset_create_version(
                str(staging), f"ep{epoch} f1={f1:.4f} {run_name}",
                delete_not_versioned_files=True)
        except Exception:
            _KAGGLE_API.dataset_create_new(str(staging))
        print(f"  ↑ best model pushed → dataset {_BEST_DS_SLUG} (ep{epoch}, f1={f1:.4f})")
    except Exception as e:
        print(f"  ⚠ upload best failed: {e}")


print(f"Checkpoint helpers ready. start_epoch = {start_epoch}")

In [ ]:
# ── Training + evaluation functions ─────────────────────────────────────────

def compute_metrics(y_true, y_pred, prefix):
    arr_t, arr_p = np.array(y_true), np.array(y_pred)
    return {
        f"{prefix}_accuracy":       float((arr_t == arr_p).mean()),
        f"{prefix}_macro_f1":       float(f1_score(arr_t, arr_p, average="macro",   zero_division=0)),
        f"{prefix}_real_f1":        float(f1_score(arr_t, arr_p, pos_label=0, average="binary", zero_division=0)),
        f"{prefix}_fake_f1":        float(f1_score(arr_t, arr_p, pos_label=1, average="binary", zero_division=0)),
        f"{prefix}_real_precision": float(precision_score(arr_t, arr_p, pos_label=0, zero_division=0)),
        f"{prefix}_real_recall":    float(recall_score(arr_t, arr_p, pos_label=0, zero_division=0)),
        f"{prefix}_fake_precision": float(precision_score(arr_t, arr_p, pos_label=1, zero_division=0)),
        f"{prefix}_fake_recall":    float(recall_score(arr_t, arr_p, pos_label=1, zero_division=0)),
    }


def train_one_epoch(epoch):
    similarity_module.train()
    detection_module.train()

    shift   = CONFIG["training"]["negative_shift"]
    min_b   = CONFIG["training"]["min_batch_size"]
    gc_val  = CONFIG["training"]["grad_clip"]
    sim_w   = CONFIG["training"]["sim_weight"]

    total_sim, total_det = 0.0, 0.0
    sim_correct, sim_total = 0, 0
    all_preds, all_labels = [], []
    n_batches, n_skipped = 0, 0

    pbar = tqdm(loaders["train"], desc=f"Ep {epoch:02d} [train]", leave=False)
    for caption, image, _ in pbar:
        if caption.size(0) < min_b:
            n_skipped += 1; continue

        caption = caption.to(DEVICE)
        image   = image.to(DEVICE)

        # ── Task 1: Similarity ───────────────────────────────────────────────
        fixed_text, matched_img, unmatched_img = prepare_sim_data(caption, image, shift)

        ta_m, ia_m, pred_sim_m = similarity_module(fixed_text, matched_img)
        ta_u, ia_u, pred_sim_u = similarity_module(fixed_text, unmatched_img)

        ta_all  = torch.cat([ta_m, ta_u], dim=0)
        ia_all  = torch.cat([ia_m, ia_u], dim=0)
        lbl_cos = torch.cat([
            torch.ones( ta_m.size(0), device=DEVICE),
            -torch.ones(ta_u.size(0), device=DEVICE),
        ], dim=0)
        lbl_cls = torch.cat([
            torch.ones( ta_m.size(0), dtype=torch.long, device=DEVICE),  # 1 = matched
            torch.zeros(ta_u.size(0), dtype=torch.long, device=DEVICE),  # 0 = unmatched
        ], dim=0)

        l_sim = loss_func_sim(ta_all, ia_all, lbl_cos)
        if not torch.isfinite(l_sim):
            n_skipped += 1; optim_sim.zero_grad(); continue

        optim_sim.zero_grad()
        (sim_w * l_sim).backward()
        torch.nn.utils.clip_grad_norm_(similarity_module.parameters(), gc_val)
        optim_sim.step()

        # Track similarity accuracy (using sim_classifier output)
        sim_pred_all = torch.cat([pred_sim_m, pred_sim_u], dim=0).argmax(1)
        sim_correct += sim_pred_all.eq(lbl_cls).sum().item()
        sim_total   += lbl_cls.size(0)

        # ── Task 2: Detection ────────────────────────────────────────────────
        det_cap, det_img, det_lbl = make_detection_batch(caption, image, shift)

        with torch.no_grad():
            ta_d, ia_d, _ = similarity_module(det_cap, det_img)

        det_logits = detection_module(det_cap, det_img, ta_d, ia_d)
        l_det = loss_func_det(det_logits, det_lbl)

        if not torch.isfinite(l_det):
            n_skipped += 1; optim_det.zero_grad(); continue

        optim_det.zero_grad()
        l_det.backward()
        torch.nn.utils.clip_grad_norm_(detection_module.parameters(), gc_val)
        optim_det.step()

        total_sim += l_sim.item()
        total_det += l_det.item()
        n_batches += 1

        all_preds.extend(det_logits.argmax(1).cpu().numpy())
        all_labels.extend(det_lbl.cpu().numpy())

        pbar.set_postfix(
            sim=f"{l_sim.item():.3f}",
            det=f"{l_det.item():.3f}",
            skip=n_skipped,
        )

    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    metrics = compute_metrics(all_labels, all_preds, "train")
    metrics.update({
        "train_loss_sim":    round(total_sim / max(1, n_batches), 4),
        "train_loss_det":    round(total_det / max(1, n_batches), 4),
        "train_sim_acc":     round(sim_correct / max(1, sim_total), 4),
        "skipped_batches":   n_skipped,
    })
    return metrics


@torch.no_grad()
def evaluate(loader, split_name):
    similarity_module.eval()
    detection_module.eval()

    shift = CONFIG["training"]["negative_shift"]
    total_loss, n_batches = 0.0, 0
    all_preds, all_labels = [], []

    for caption, image, _ in tqdm(loader, desc=f"  [{split_name}]", leave=False):
        caption = caption.to(DEVICE)
        image   = image.to(DEVICE)

        det_cap, det_img, det_lbl = make_detection_batch(caption, image, shift)
        ta_d, ia_d, _ = similarity_module(det_cap, det_img)
        det_logits    = detection_module(det_cap, det_img, ta_d, ia_d)
        l = loss_func_det(det_logits, det_lbl)

        if torch.isfinite(l):
            total_loss += l.item()
        n_batches += 1

        all_preds.extend(det_logits.argmax(1).cpu().numpy())
        all_labels.extend(det_lbl.cpu().numpy())

    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    metrics = compute_metrics(all_labels, all_preds, split_name)
    metrics[f"{split_name}_loss"]     = round(total_loss / max(1, n_batches), 4)
    metrics["confusion_matrix"]       = confusion_matrix(all_labels, all_preds, labels=[0, 1]).tolist()
    return metrics


print("Training functions defined.")

In [ ]:
# ── Run Training ─────────────────────────────────────────────────────────────
history = []
best_val_f1  = -1.0
best_val_acc = -1.0
best_epoch   = -1
epochs_no_improve = 0
best_ckpt_path = run_dir / "best_macro_f1.pth"
acc_ckpt_path  = run_dir / "best_accuracy.pth"
CKPT_EVERY = CONFIG["checkpointing"]["checkpoint_every"]
PATIENCE   = CONFIG["training"]["early_stopping_patience"]

print(f"Training CAFE for {CONFIG['training']['max_epochs']} epochs (early stopping patience={PATIENCE})")
print(f"Device: {DEVICE}")
print()

try:
    for epoch in range(start_epoch, CONFIG["training"]["max_epochs"]):
        tr = train_one_epoch(epoch)
        va = evaluate(loaders["dev"], "val")
        sched_sim.step()
        sched_det.step()

        rec = {
            "epoch":           epoch,
            "train_loss_sim":  tr["train_loss_sim"],
            "train_loss_det":  tr["train_loss_det"],
            "train_sim_acc":   tr["train_sim_acc"],
            "train_accuracy":  tr["train_accuracy"],
            "train_macro_f1":  tr["train_macro_f1"],
            "val_loss":        va["val_loss"],
            "val_accuracy":    va["val_accuracy"],
            "val_macro_f1":    va["val_macro_f1"],
            "val_real_f1":     va["val_real_f1"],
            "val_fake_f1":     va["val_fake_f1"],
            "lr_sim":          optim_sim.param_groups[0]["lr"],
            "lr_det":          optim_det.param_groups[0]["lr"],
            "skipped_batches": tr["skipped_batches"],
        }
        history.append(rec)

        # Periodic checkpoint
        if (epoch + 1) % CKPT_EVERY == 0:
            save_checkpoint(run_dir / f"ckpt_ep{epoch}.pth", epoch, history, rec)

        # Best val macro-F1
        if va["val_macro_f1"] > best_val_f1:
            best_val_f1 = va["val_macro_f1"]; best_epoch = epoch
            epochs_no_improve = 0
            save_checkpoint(best_ckpt_path, epoch, history, rec)
            kaggle_upload_best(best_ckpt_path, epoch, best_val_f1)
            print(f"  ★ best val_f1={best_val_f1:.4f} @ epoch {epoch} → {best_ckpt_path.name}")
        else:
            epochs_no_improve += 1

        # Best val accuracy (secondary)
        if va["val_accuracy"] > best_val_acc:
            best_val_acc = va["val_accuracy"]
            save_checkpoint(acc_ckpt_path, epoch, history, rec)

        with open(run_dir / "history.json", "w") as f: json.dump(history, f, indent=2)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        print(
            f"Ep {epoch:02d} | "
            f"sim={rec['train_loss_sim']:.3f} det={rec['train_loss_det']:.3f} | "
            f"sim_acc={rec['train_sim_acc']:.3f} tr_f1={rec['train_macro_f1']:.3f} | "
            f"val_acc={rec['val_accuracy']:.3f} val_f1={rec['val_macro_f1']:.3f} "
            f"(real={rec['val_real_f1']:.3f} fake={rec['val_fake_f1']:.3f}) | "
            f"skip={rec['skipped_batches']} | no_improve={epochs_no_improve}/{PATIENCE}"
        )

        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping: no val_macro_f1 improvement for {PATIENCE} epochs "
                  f"(best={best_val_f1:.4f} @ epoch {best_epoch}).")
            break

except KeyboardInterrupt:
    _int = run_dir / f"interrupted_ep{epoch}.pth"
    save_checkpoint(_int, epoch, history, {})
    print(f"Interrupted → {_int}")
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError(f"CUDA OOM — lower batch_size or enable smoke_test.\n{e}")
    raise

print(f"\nDone. Best val_macro_f1={best_val_f1:.4f} at epoch {best_epoch}.")

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────
df = pd.DataFrame(history)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle(f"CAFE — {run_name}", fontsize=13)

# Task 1: Similarity loss
ax = axes[0, 0]
ax.plot(df.epoch, df.train_loss_sim, label="L_SIM (train)", color="royalblue")
ax.set_title("Task 1: Similarity Loss"); ax.legend(); ax.set_xlabel("Epoch")

# Task 2: Detection loss
ax = axes[0, 1]
ax.plot(df.epoch, df.train_loss_det, label="L_DET (train)", color="coral")
ax.plot(df.epoch, df.val_loss,       label="L_DET (val)",   color="coral", linestyle="--")
ax.set_title("Task 2: Detection Loss"); ax.legend(); ax.set_xlabel("Epoch")

# Accuracy
ax = axes[0, 2]
ax.plot(df.epoch, df.train_accuracy, label="Train", color="steelblue")
ax.plot(df.epoch, df.val_accuracy,   label="Val",   color="coral")
ax.set_title("Detection Accuracy"); ax.legend(); ax.set_xlabel("Epoch")

# Macro F1
ax = axes[1, 0]
ax.plot(df.epoch, df.train_macro_f1, label="Train macro-F1", color="steelblue")
ax.plot(df.epoch, df.val_macro_f1,   label="Val macro-F1",   color="coral")
ax.set_title("Macro F1"); ax.legend(); ax.set_xlabel("Epoch")

# Per-class F1
ax = axes[1, 1]
ax.plot(df.epoch, df.val_real_f1, label="Val Real F1 (0)", color="green")
ax.plot(df.epoch, df.val_fake_f1, label="Val Fake F1 (1)", color="red")
ax.set_title("Val Per-Class F1"); ax.legend(); ax.set_xlabel("Epoch")

# Similarity accuracy
ax = axes[1, 2]
ax.plot(df.epoch, df.train_sim_acc, label="Sim Acc (train)", color="purple")
ax.set_title("Task 1: Similarity Accuracy"); ax.legend(); ax.set_xlabel("Epoch")

plt.tight_layout()
fig.savefig(run_dir / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {run_dir / 'training_curves.png'}")

In [ ]:
# ── Final evaluation on test set ─────────────────────────────────────────────
print(f"Loading best checkpoint: {best_ckpt_path}")
load_checkpoint(best_ckpt_path)

test_metrics = evaluate(loaders["test"], "test")
cm = np.array(test_metrics.pop("confusion_matrix"))

print("\n── Test Results ────────────────────────────────────────")
for k, v in test_metrics.items():
    if isinstance(v, float): print(f"  {k:<28} {v:.4f}")

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Real (0)", "Fake (1)"],
            yticklabels=["Real (0)", "Fake (1)"], ax=ax)
ax.set_title(f"CAFE — Confusion Matrix (test)\nMacro-F1 = {test_metrics['test_macro_f1']:.4f}")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
fig.savefig(run_dir / "confusion_matrix_test.png", dpi=120, bbox_inches="tight")
plt.show()

# Save final test results
test_metrics["best_epoch"] = best_epoch
test_metrics["run_name"]   = run_name
(run_dir / "test_results.json").write_text(json.dumps(test_metrics, indent=2))
print(f"\nTest results saved → {run_dir / 'test_results.json'}")
print(f"Best checkpoint: {best_ckpt_path}")